<h1 style=\"text-align: center; font-size: 50px;\">  Text Generation with Neural Networks and Torch MLflow Integration</h1>

# Notebook Overview
- Start Execution
- User Constants
- Install and Import Libraries
- Configure Settings
- Verify Assets
- Logging Model to MLflow
- Fetching the Latest Model Version from MLflow
- Loading the Model and Running Inference

In [1]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 10.5 ms, sys: 6.71 ms, total: 17.2 ms
Wall time: 655 ms


In [2]:
MIN_TOTAL_RAM_GB = 16
MIN_TOTAL_VRAM_GB = 4


from ai_studio_blueprint_kit.memory_guard import run_memory_check_notebook


run_memory_check_notebook(
    min_total_ram_gb=MIN_TOTAL_RAM_GB,
    min_total_vram_gb=MIN_TOTAL_VRAM_GB,
)

## Start Execution

In [1]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [2]:
start_time = time.time()  

logger.info("Notebook execution started.")

2026-04-15 04:08:31 - INFO - Notebook execution started.


## Install and Import Libraries

In [4]:
# -----------------------------
# Standard library imports
# -----------------------------
import os                   # Operating system utilities (paths, env vars, etc.)
import sys                  # Python runtime environment manipulation
import time                 # Time-related utilities
import warnings             # Warning control and message handling
from datetime import datetime  # Date and time handling
from pathlib import Path     # Object-oriented filesystem paths

# -----------------------------
# Third-party imports
# -----------------------------
import numpy as np           # Numerical computations and arrays
import pandas as pd          # Data manipulation and analysis
import torch                 # PyTorch deep learning framework
import torch.nn.functional as F  # Functional API for neural network operations
from torch import nn          # Neural network layers and modules

import mlflow                 # ML lifecycle management and experiment tracking
from mlflow.models import ModelSignature  # MLflow model signature definition
from mlflow.types.schema import ColSpec, Schema  # MLflow schema utilities

# -----------------------------
# Local imports
# -----------------------------
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.mlflow import Logger

from src.utils import (
    load_config,
    download_from_s3_uri 
)

## User Constants

In [5]:
INITIAL_WORD = 'Love'
SIZE = 100

## Configure Settings

In [6]:
torch.manual_seed(0)

In [7]:
warnings.filterwarnings("ignore")

In [8]:
# ------------------------ Define global experiment and run names to be used throughout the notebook ------------------------
RUN_NAME = "RNN Text Generation"
MODEL_NAME = "dict_torch_rnn_model"
TORCH_MODEL = "dict_torch_rnn_model.pt"
EXPERIMENT_NAME = "Shakespeare Text Generation"
REGISTER_NAME = "Shakespeare_Model"

# ------------------------ Remote asset URIs ------------------------
S3_BASE = "s3://149536453923-hpaistudio-public-assets/AI-Blueprints/deep-learning/text-generation-with-rnn"
MODEL_URI = f"{S3_BASE}/{TORCH_MODEL}"
DATA_NAME = "shakespeare.txt"
DATA_URI = f"{S3_BASE}/{DATA_NAME}"

# ------------------------ Local paths ------------------------
ROOT = Path("..").resolve()
MODELS_PATH = ROOT / "models"
DATA_PATH = ROOT / "data" / DATA_NAME
MODEL_DECODER_PATH = ROOT / "models" / "decoder.pt"
MODEL_ENCODER_PATH = ROOT / "models" / "encoder.pt"
MODEL_STATE_PATH = ROOT / "models" / TORCH_MODEL
MODEL_PATH = ROOT / "models" / 'dict_torch_rnn_model.pt'
DEMO_FOLDER = ROOT / "demo"
CONFIG_PATH = ROOT / "configs" / "config.yaml"

# ------------------------ MLflow tracking ------------------------
# If your environment / platform uses a local MLflow path, set here.
# Compatible with Phoenix MLflow if mounted at this path.
MLFLOW_TRACKING_URI = "/phoenix/mlflow"

## Verify Assets

In [9]:
def log_asset_status(asset_path: str, asset_name: str, success_message: str, failure_message: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
        success_message (str): Message to log if asset exists.
        failure_message (str): Message to log if asset does not exist.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured. {success_message}")
    else:
        logger.error(f"{asset_name} is not properly configured. {failure_message}")
        
log_asset_status(
    asset_path=DATA_PATH,
    asset_name="Shakespeare text",
    success_message="",
    failure_message="Please run the 'run-workflow' notebook first and check if data folder was properly downloaded in your project on AI Studio."
)

log_asset_status(
    asset_path=MODEL_DECODER_PATH ,
    asset_name="Decoder model",
    success_message="",
    failure_message="Please check if models folder was properly configured in your project on AI Studio."
)

log_asset_status(
    asset_path=MODEL_ENCODER_PATH,
    asset_name="Encoder model",
    success_message="",
    failure_message="Please check if models folder was properly configured in your project on AI Studio."
)

log_asset_status(
    asset_path=MODEL_STATE_PATH,
    asset_name="Rnn model",
    success_message="",
    failure_message="Please if models folder was properly downloaded in your project on AI Studio."
)
log_asset_status(
    asset_path=DEMO_FOLDER,
    asset_name="demo",
    success_message="",
    failure_message="Please check if demo folder was properly configured in your project on AI Studio."
)
log_asset_status(
    asset_path=CONFIG_PATH,
    asset_name="config",
    success_message="",
    failure_message="Please check if config file was properly configured in your project on AI Studio."
)

2026-04-15 04:08:34 - INFO - Shakespeare text is properly configured. 
2026-04-15 04:08:34 - INFO - Decoder model is properly configured. 
2026-04-15 04:08:34 - INFO - Encoder model is properly configured. 
2026-04-15 04:08:34 - INFO - Rnn model is properly configured. 
2026-04-15 04:08:34 - INFO - demo is properly configured. 
2026-04-15 04:08:34 - INFO - config is properly configured. 


## Logging Model to MLflow

In [10]:
# Define input/output schema for the RNN text generation model
input_schema = Schema([
    ColSpec("string", "initial_word"),
    ColSpec("long", "size")
])

output_schema = Schema([
    ColSpec("string", "generated_text")
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema)
logger.info("Model signature created successfully")

2026-04-15 04:08:34 - INFO - Model signature created successfully


In [11]:
mlflow.set_tracking_uri('/phoenix/mlflow')
mlflow.set_experiment(experiment_name= EXPERIMENT_NAME)

<Experiment: artifact_location='/phoenix/mlflow/437501764375616290', creation_time=1776225737336, experiment_id='437501764375616290', last_update_time=1776225737336, lifecycle_stage='active', name='Shakespeare Text Generation', tags={}, workspace='default'>

In [12]:
model_state_dict = MODEL_PATH
register_name = REGISTER_NAME 

In [13]:
with mlflow.start_run(run_name = RUN_NAME) as run:
    logger.info(f"Run's Artifact URI: {run.info.artifact_uri}")
    
    # Use new Logger with models-from-code approach
    Logger.log_model(
        signature=signature,
        model_state_dict_path=MODEL_PATH,
        decoder_path=MODEL_DECODER_PATH,
        encoder_path=MODEL_ENCODER_PATH,
        artifact_path=REGISTER_NAME,
        config_path=CONFIG_PATH,
        data_path=DATA_PATH,
        demo_folder=DEMO_FOLDER
    )
    
    mlflow.register_model(model_uri = f"runs:/{run.info.run_id}/{REGISTER_NAME}", name=register_name)

2026-04-15 04:08:35 - INFO - Run's Artifact URI: /phoenix/mlflow/437501764375616290/457ca885e99e4467b1b9b596e2755ab7/artifacts
Successfully registered model 'Shakespeare_Model'.
2026/04/15 04:08:40 WARNING mlflow.tracking._model_registry.fluent: Run with id 457ca885e99e4467b1b9b596e2755ab7 has no artifacts at artifact path 'Shakespeare_Model', registering model based on models:/m-c31b93ac12ae45db8a8195152cd219e2 instead
Created version '1' of model 'Shakespeare_Model'.


## Fetching the Latest Model Version from MLflow

In [14]:
client = mlflow.MlflowClient()
model_metadata = client.get_latest_versions(register_name, stages=["None"])
latest_model_version = model_metadata[0].version
latest_model_version

1

## Loading the Model and Running Inference

In [15]:
loaded = mlflow.pyfunc.load_model(model_uri=f"models:/{REGISTER_NAME}/{latest_model_version}")
# IMPORTANT: pyfunc expects a DataFrame input per our signature
in_df = pd.DataFrame({"initial_word": [INITIAL_WORD], "size": [SIZE]})
result_df = loaded.predict(in_df)
print(result_df["generated_text"].iloc[0])

Lovers, Caesar's hand, and his house

  KING HENRY. I am some the commission.
  CASCA. Whether the man is


In [16]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")

2026-04-15 04:08:43 - INFO - ⏱️ Total execution time: 0m 11.65s


In [17]:
print("Notebook execution completed successfully.")

Notebook execution completed successfully.


Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).